# 03 — RAG Pipeline: LangChain Chain + Memory + Testing

**Goal:** Load the vector store from disk, initialise `FinancialRAGChain`, and run 20 test questions to validate the full pipeline.

**API cost:** ~$0.05 USD (20 questions × ~250 tokens avg)

## 1. Load Vector Store from Disk

In [1]:
import sys
import time
from pathlib import Path

import pandas as pd

ROOT = Path().resolve().parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.vectorstore import load_vectorstore
from src.rag_chain import FinancialRAGChain

VECTORSTORE_DIR = str(ROOT / 'vectorstore')

vs = load_vectorstore(VECTORSTORE_DIR)
print('Vector store loaded ✅')

Failed to send telemetry event ClientStartEvent: capture() takes 1 positional argument but 3 were given
Failed to send telemetry event ClientCreateCollectionEvent: capture() takes 1 positional argument but 3 were given


Loaded vectorstore — 1183 documents from /Users/nicolaszuleta95/code_nz/financial-rag-assistant/vectorstore/
Vector store loaded ✅


## 2. Initialise FinancialRAGChain

In [2]:
chain = FinancialRAGChain(vs)
print('FinancialRAGChain initialised ✅')
print('LLM  : gpt-4o-mini (temperature=0, max_tokens=500)')
print('Top-k: 4 chunks per query')

FinancialRAGChain initialised ✅
LLM  : gpt-4o-mini (temperature=0, max_tokens=500)
Top-k: 4 chunks per query


## 3. Basic Test — 5 Simple Questions

In [3]:
def ask_and_print(question, chain_obj):
    t0 = time.time()
    result = chain_obj.ask(question)
    elapsed = time.time() - t0
    print(f'Q: {question}')
    print(f'A: {result["answer"][:400]}')
    print(f'   Sources: {[(s["bank"], s["page"]) for s in result["sources"]]}')
    print(f'   Time: {elapsed:.1f}s  |  Tokens: {result["tokens_used"]}')
    print()
    return result

basic_questions = [
    "What was JPMorgan's total net revenue in 2023?",
    "What is Bank of America's CET1 capital ratio?",
    "How many employees does Wells Fargo have?",
    "What are the main business segments of JPMorgan Chase?",
    "What was the provision for credit losses at Bank of America?",
]

chain.clear_memory()
for q in basic_questions:
    ask_and_print(q, chain)

Failed to send telemetry event CollectionQueryEvent: capture() takes 1 positional argument but 3 were given


Q: What was JPMorgan's total net revenue in 2023?
A: JPMorgan Chase reported total net revenue of $158.1 billion for the year ended December 31, 2023, which represents an increase of 23% compared to the previous year (JPMorgan Chase & Co., 2023, page 135).
   Sources: [('JPMorgan Chase', 203), ('JPMorgan Chase', 135), ('JPMorgan Chase', 671), ('JPMorgan Chase', 171)]
   Time: 3.5s  |  Tokens: 42

Q: What is Bank of America's CET1 capital ratio?
A: - **Bank of America Corporation (December 31, 2023)**:
  - Common Equity Tier 1 (CET1) capital ratio: **11.8%** under the Standardized approach (Bank of America, 2023, p. 48).
  
- **JPMorgan Chase & Co. (December 31, 2023)**:
  - CET1 capital ratio: **15.0%** under the Standardized approach (JPMorgan, 2023, p. 620).

In summary, Bank of America's CET1 capital ratio of 11.8% is lower than JPMorgan
   Sources: [('Bank of America', 353), ('Bank of America', 135), ('Bank of America', 133), ('JPMorgan Chase', 620)]
   Time: 4.5s  |  Tokens: 75

Q

## 4. Source Citation Test

Verify that every response includes correct bank name and page number.

In [4]:
chain.clear_memory()
result = chain.ask("What were JPMorgan's risk factors related to interest rates in 2023?")

print('ANSWER:')
print(result['answer'])
print()
print('SOURCES RETRIEVED:')
for i, src in enumerate(result['sources'], 1):
    print(f'{i}. [{src["bank"]} {src["year"]} · Page {src["page"]}]')
    print(f'   {src["text"][:200]}...')
    print()

ANSWER:
JPMorgan Chase identified several risk factors related to interest rates in its 2023 Form 10-K:

- **High Interest Rates:**
  - Potential for increased net interest income.
  - Risk of fewer originations of commercial and residential real estate loans.
  - Possibility of losses on underwriting exposures or client-specific downgrades.
  - Increased allowance for credit losses and net charge-offs due to higher financing costs for clients.
  - Risk of losing deposits if customers perceive better interest rates offered by competitors.
  - Potential losses on available-for-sale (AFS) securities in the investment portfolio.
  - Lower net interest income if central banks raise rates more quickly than anticipated, leading to misalignment in pricing.
  - Reduced liquidity in financial markets and higher funding costs.
  - Negative impact on payment performance for variable-rate loans, leading to increased operational costs related to servicing delinquent loans (JPMorgan Chase & Co., 202

## 5. Conversational Memory Test

Follow-up questions should leverage conversation history for accurate retrieval.

In [5]:
chain.clear_memory()

conversation = [
    "What was JPMorgan's net interest income in 2023?",
    "How does that compare to the previous year?",
    "And what about Bank of America's net interest income?",
]

for turn, q in enumerate(conversation, 1):
    print(f'--- Turn {turn} ---')
    ask_and_print(q, chain)

--- Turn 1 ---
Q: What was JPMorgan's net interest income in 2023?
A: JPMorgan's net interest income in 2023 was $89.3 billion, reflecting a significant increase from the previous year (JPMorgan Chase & Co., 2023, page 135).
   Sources: [('JPMorgan Chase', 203), ('JPMorgan Chase', 135), ('JPMorgan Chase', 142), ('JPMorgan Chase', 671)]
   Time: 1.8s  |  Tokens: 32

--- Turn 2 ---
Q: How does that compare to the previous year?
A: JPMorgan Chase reported net interest income of $89.3 billion for the year ended December 31, 2023, which represents an increase of 34% compared to the net interest income of $66.7 billion for the year ended December 31, 2022 (JPMorgan Chase & Co., 2023, page 135).
   Sources: [('JPMorgan Chase', 203), ('JPMorgan Chase', 135), ('JPMorgan Chase', 679), ('JPMorgan Chase', 162)]
   Time: 3.1s  |  Tokens: 53

--- Turn 3 ---
Q: And what about Bank of America's net interest income?
A: Bank of America's net interest income in 2023 was $56.9 billion, an increase of $4.5

## 6. Metadata Filter Test

Query with `bank_filter` to restrict retrieval to a single institution.

In [6]:
question = "What is the digital banking strategy?"

for bank in ['JPMorgan Chase', 'Bank of America', 'Wells Fargo']:
    bank_chain = FinancialRAGChain(vs, bank_filter=bank)
    result = bank_chain.ask(question)
    banks_in_sources = {s['bank'] for s in result['sources']}
    print(f'Filter={bank:20s}  Sources banks={banks_in_sources}')
    print(f'  {result["answer"][:200]}...')
    print()

Filter=JPMorgan Chase        Sources banks={'JPMorgan Chase'}
  I don't have that information in the available documents....

Filter=Bank of America       Sources banks={'Bank of America'}
  I don't have that information in the available documents....

Filter=Wells Fargo           Sources banks={'Wells Fargo'}
  I don't have that information in the available documents....



## 7. "I Don't Know" Test

The system must refuse to answer questions about information not present in the documents.

In [7]:
chain.clear_memory()

out_of_scope = [
    "What is the current stock price of JPMorgan?",
    "Who will be the CEO of Bank of America in 2030?",
]

for q in out_of_scope:
    result = chain.ask(q)
    print(f'Q: {q}')
    print(f'A: {result["answer"]}')
    refused = "don't have" in result['answer'].lower() or "not in" in result['answer'].lower()
    print(f'   Correctly refused: {"✅" if refused else "❌"}')
    print()

Q: What is the current stock price of JPMorgan?
A: I don't have that information in the available documents.
   Correctly refused: ✅

Q: Who will be the CEO of Bank of America in 2030?
A: I don't have that information in the available documents.
   Correctly refused: ✅



## 8. Cross-bank Comparison Test

In [8]:
chain.clear_memory()

comparison_questions = [
    "Compare the capital ratios of JPMorgan, Bank of America, and Wells Fargo",
    "Which bank reported the highest return on equity?",
    "How do the risk management approaches differ across the three banks?",
]

for q in comparison_questions:
    ask_and_print(q, chain)

Q: Compare the capital ratios of JPMorgan, Bank of America, and Wells Fargo
A: I don't have that information in the available documents.
   Sources: [('JPMorgan Chase', 619), ('JPMorgan Chase', 240), ('JPMorgan Chase', 618), ('JPMorgan Chase', 620)]
   Time: 1.1s  |  Tokens: 21

Q: Which bank reported the highest return on equity?
A: I don't have that information in the available documents.
   Sources: [('JPMorgan Chase', 246), ('JPMorgan Chase', 660), ('JPMorgan Chase', 131), ('JPMorgan Chase', 135)]
   Time: 1.6s  |  Tokens: 17

Q: How do the risk management approaches differ across the three banks?
A: I don't have that information in the available documents.
   Sources: [('JPMorgan Chase', 214), ('JPMorgan Chase', 19), ('JPMorgan Chase', 28), ('JPMorgan Chase', 31)]
   Time: 2.1s  |  Tokens: 20



## 9. Latency Analysis

In [9]:
import statistics

latency_questions = [
    "What is JPMorgan's return on equity?",
    "Describe Wells Fargo's risk factors",
    "What are Bank of America's revenue streams?",
    "How does JPMorgan manage credit risk?",
    "What is the regulatory environment for large US banks?",
]

latencies = []
chain.clear_memory()

for q in latency_questions:
    t0 = time.time()
    chain.ask(q)
    latencies.append(time.time() - t0)

print(f'Latency over {len(latencies)} questions:')
print(f'  Min   : {min(latencies):.2f}s')
print(f'  Max   : {max(latencies):.2f}s')
print(f'  Mean  : {statistics.mean(latencies):.2f}s')
print(f'  Median: {statistics.median(latencies):.2f}s')

Latency over 5 questions:
  Min   : 1.60s
  Max   : 10.18s
  Mean  : 5.29s
  Median: 4.87s


## 10. Token Cost Analysis

In [10]:
cost_questions = [
    "What are JPMorgan's main revenue drivers?",
    "Explain Wells Fargo's credit loss provision",
    "What is Bank of America's loan portfolio composition?",
    "Describe regulatory challenges facing JPMorgan",
    "How did rising interest rates affect net interest margin?",
]

chain.clear_memory()
token_counts = []
for q in cost_questions:
    result = chain.ask(q)
    token_counts.append(result['tokens_used'])

avg_tokens = sum(token_counts) / len(token_counts)
cost_per_question = (avg_tokens / 1_000_000) * 0.15  # gpt-4o-mini input price

print(f'Avg tokens per question: {avg_tokens:.0f}')
print(f'Estimated cost per question: ${cost_per_question:.5f} USD')
print(f'100 questions cost: ${cost_per_question * 100:.3f} USD')

Avg tokens per question: 152
Estimated cost per question: $0.00002 USD
100 questions cost: $0.002 USD
